# Huấn luyện trên Kaggle

**Cài đặt Notebook:** Accelerator = **GPU T4 x2**, Internet = **On**, Add Data = `maitam/vietnamese-traffic-signs`.

Kaggle đã cài sẵn torch/torchvision — **không** cài lại torch; chỉ cài các thư viện còn thiếu (bước 1) và ghi lại phiên bản đang dùng để đưa vào báo cáo.

## 1. Cài thư viện còn thiếu

In [ ]:
!pip install -q ultralytics==8.4.175 torchmetrics==1.9.0 pycocotools==2.0.11 PyYAML==6.0.3
!pip list 2>/dev/null | grep -iE "^(torch|torchvision|ultralytics|torchmetrics|numpy|opencv)"

## 2. Lấy mã nguồn

In [ ]:
!git clone -b data-yolo https://github.com/ThanhBinh2006/VN-TSD-Voice-Alert.git /kaggle/working/VN-TSD-Voice-Alert
%cd /kaggle/working/VN-TSD-Voice-Alert


## 3. Tiền xử lý offline (Thành viên 1)

In [ ]:
!python -m tools.run_preprocess --data-root /kaggle/input/vietnamese-traffic-signs --runtime configs/runtime/kaggle.yaml

## 4. Huấn luyện Faster R-CNN (Thành viên 2) — DDP 2 GPU

In [ ]:
!torchrun --nproc_per_node=2 -m tools.run_train --pipeline faster_rcnn --runtime configs/runtime/kaggle.yaml --run-name faster_rcnn_s42

## 5. Huấn luyện YOLO26m (Thành viên 1)

Ultralytics tự chạy DDP khi `device=[0, 1]`. TODO: Thành viên 1 xác nhận cách chạy đa GPU trên Kaggle (chạy trực tiếp như dưới đây, hay bọc bằng torchrun) rồi sửa lại ô này.

In [ ]:
!python -m tools.run_train --pipeline yolo --runtime configs/runtime/kaggle.yaml --run-name yolo26m_s42

## 6. Đánh giá trên tập Test (Thành viên 2)

Thay `<run_id>` bằng tên thư mục trong `/kaggle/working/outputs/<pipeline>/`.

In [ ]:
!python -m tools.run_eval --pipeline yolo --runtime configs/runtime/kaggle.yaml --run-dir /kaggle/working/outputs/yolo/<run_id>
!python -m tools.run_eval --pipeline faster_rcnn --runtime configs/runtime/kaggle.yaml --run-dir /kaggle/working/outputs/faster_rcnn/<run_id>

## 7. Đóng gói để tải về (Output của notebook)

In [ ]:
!python -m tools.pack_outputs --run-dir /kaggle/working/outputs/yolo/<run_id>
!python -m tools.pack_outputs --run-dir /kaggle/working/outputs/faster_rcnn/<run_id>